# 24_Image 가중치로 추론 수행하기

## 학습목표 
- 1. 학습 이후 얻은 가중치 파일을 활용하여 새로운 데이터에 대한 추론을 수행합니다.   

In [1]:
import os #로컬 컴퓨터에서 데이터셋을 불러오기 위해
import cv2 # 시각화 해서 표현하기 위함
import numpy as np #이미지 픽셀 데이터를 numpy 형태로 표현함
import json #개별 라벨을 읽어들어오게 함

import torch #훈련
from torch.utils.data import Dataset, DataLoader #커스텀 데이터셋을 만들기 위함
from torchvision import transforms #torchvision -> 딥러닝(이미지) 수행하는 클래스 이름 #transform 이미지를 조절
from PIL import Image #-> 이미지를 표현할 수 있는 파이썬 라이브러리 
import matplotlib.pyplot as plt
import itertools

In [2]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"DEVICE : {device}")

DEVICE : cuda


In [ ]:
import torch
import torchsummary
import torchvision
import torchvision.models.detection

from tqdm import tqdm  # 학습 진행률 시각화
from torch import nn
import torchvision.transforms.functional as F

# 모델 로드 (사전학습된 Faster R-CNN)
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()  # 추론 모드 설정

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

### 가중치 파일 불러오기

In [10]:
model.load_state_dict(torch.load("fasterrcnn_trained.pth", map_location=device))

<All keys matched successfully>

In [ ]:
# 이미지 불러오기
image_path = "C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/YOLO/Images/valid/upper_29253.png"
image = Image.open(image_path).convert("RGB")
transform = transforms.Compose([
                        #전처리의 다양한 종류를 여기서 적용해준다.
                        transforms.Resize((224, 224)),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                    ])

#image에 transform을 적용
input_tensor = transform(image)           # [3, 224, 224]
input_tensor = input_tensor.unsqueeze(0)  # [1, 3, 224, 224] → 배치 차원 추가

# 이미지 전처리
img_tensor = input_tensor.to(device)  # [C, H, W] 형태로 변환

In [ ]:
#추론
with torch.no_grad():
    prediction = model(img_tensor)

#결과 출력
for idx in range(len(prediction[0]['boxes'])):
    box = prediction[0]['boxes'][idx].cpu().numpy()
    label = prediction[0]['labels'][idx].item()
    score = prediction[0]['scores'][idx].item()
    print(f"Object {idx}: Label={label}, Score={score:.2f}, Box={box}")

In [15]:
prediction

[{'boxes': tensor([], device='cuda:0', size=(0, 4)),
  'labels': tensor([], device='cuda:0', dtype=torch.int64),
  'scores': tensor([], device='cuda:0')}]

데이터 준비 -> 데이터의 변형 -> 모델 세팅 -> 학습 -> 추론